In [3]:
import pandas as pd

def filter_homologues(df_train, mmseqs_result_path):
    """
    MMseqs2 결과에 포함된 유사 서열을 훈련 데이터프레임에서 제거합니다.
    """
    # 1. MMseqs2 결과 로드 (m8 포맷: query, target, identity, ...)
    # 보통 2번째 컬럼(index 1)이 유사한 Target(훈련셋) ID입니다.
    results = pd.read_csv(mmseqs_result_path, sep='\t', header=None)
    similar_ids = set(results[1].unique())
    
    print(f"검색된 유사 서열(Identity > 25%) 개수: {len(similar_ids)}")
    
    # 2. Key(PDB_ID_Chain 형식 등)를 기준으로 필터링
    # 기존 코드에서 생성했던 'Key' 컬럼을 활용합니다.
    df_filtered = df_train[~df_train['Key'].isin(similar_ids)].copy()
    
    removed_count = len(df_train) - len(df_filtered)
    print(f"제거된 훈련 데이터 행 수: {removed_count}")
    print(f"최종 남은 데이터 수: {len(df_filtered)}")
    
    return df_filtered

# --- 실행 부분 ---
# MMseqs2 결과 파일 경로
mmseqs_res = r"C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\FireProtDB\result_25.m8"

df_mega = pd.read_csv(r"C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\FireProtDB\Mega.tsv", sep="\t")
df_notMega = pd.read_csv(r"C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\FireProtDB\notMega.tsv", sep="\t")

# MegaScale 데이터 필터링
df_mega_filtered = filter_homologues(df_mega, mmseqs_res)

# notMega 데이터 필터링 (동일한 결과 파일 사용)
df_notMega_filtered = filter_homologues(df_notMega, mmseqs_res)

검색된 유사 서열(Identity > 25%) 개수: 20
제거된 훈련 데이터 행 수: 0
최종 남은 데이터 수: 301709
검색된 유사 서열(Identity > 25%) 개수: 20
제거된 훈련 데이터 행 수: 166
최종 남은 데이터 수: 3735


In [9]:
df_combined_train = pd.concat([df_mega_filtered, df_notMega_filtered], axis=0, ignore_index=True)


In [10]:
df_combined_train

,WT,MutPos,Mut,DDG,PDB_ID,FILE_PATH,Chain,SeqPos,Key
0,A,45,C,0.382630,1A32,1A32.pdb,A,45,1A32_A
1,A,45,D,-0.012520,1A32,1A32.pdb,A,45,1A32_A
2,A,45,E,0.259534,1A32,1A32.pdb,A,45,1A32_A
3,A,45,F,-0.290754,1A32,1A32.pdb,A,45,1A32_A
4,A,45,G,-0.221574,1A32,1A32.pdb,A,45,1A32_A
...,...,...,...,...,...,...,...,...,...
305439,T,68,A,0.300000,R9S082,R9S082_AF.pdb,A,68,R9S082_A
305440,V,115,A,1.400000,R9S082,R9S082_AF.pdb,A,115,R9S082_A
305441,V,14,A,0.000000,R9S082,R9S082_AF.pdb,A,14,R9S082_A
305442,V,69,T,0.400000,R9S082,R9S082_AF.pdb,A,69,R9S082_A


In [11]:
initial_count = len(df_combined_train)
df_combined_train = df_combined_train.drop_duplicates(subset=['PDB_ID', 'Chain', 'MutPos', 'WT', 'Mut'])
final_count = len(df_combined_train)

if initial_count != final_count:
    print(f"주의: Mega와 notMega 사이의 중복 데이터 {initial_count - final_count}건을 제거했습니다.")

In [12]:
df_combined_train.to_csv("FireProt_s669_train.tsv", sep='\t', index=False)